# BERTopic Output Exploration

**Latest version:** May 2026

## Notebook overview

This notebook explores the outputs of `3_topic_modelling.ipynb`. It covers:

| Step | What happens | Output |
|------|-------------|--------|
| 1 | Imports & settings | - |
| 2 | Load data (documents, embeddings, topic model, results) | - |
| 3 | Counts, topic table, size distribution | `topic_size_distribution.png` |
| 4 | Drill into any topic: top words, sample docs | - |
| 5 | Characterise remaining outlier documents | - |
| 6 | Cosine-similarity check: do outliers belong elsewhere? | - |
| 7 | Gini concentration, size anomalies | - |
| 8 | Jaccard overlap across topics to detect splits | - |
| 9 | Built-in BERTopic interactive plots & visualizations | - |
| 10 | Topic coherence (C_v): Per-topic semantic quality metric | `topic_coherence_distribution.png` |
| 11 | Topic diversity: Unique keyword ratio across all topics | - |

**Prerequisites:** Run `3_topic_modelling.ipynb` first.  
**Optional inputs:** A manually labelled topic table (`topic_info_labelled.xlsx`) unlocks label-based sections. All sections still run even if optional files are absent.

---
# 1. Imports & settings

In [1]:
import pickle
import random
import warnings
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import entropy, ttest_ind, levene
from sklearn.metrics.pairwise import cosine_similarity
from bertopic import BERTopic

# Gensim is only needed for C_v coherence (Section 10)
try:
    from gensim.corpora import Dictionary
    from gensim.models.coherencemodel import CoherenceModel
    GENSIM_AVAILABLE = True
except ImportError:
    GENSIM_AVAILABLE = False
    warnings.warn("gensim not installed – Section 10 (C_v coherence) will be skipped.")

# squarify is only needed for the treemap (Section 9)
try:
    import squarify
    SQUARIFY_AVAILABLE = True
except ImportError:
    SQUARIFY_AVAILABLE = False
    warnings.warn("squarify not installed – treemap will be skipped. Install with: pip install squarify")

# Project config
from config import *

## ⚙️ User settings

**Edit this cell before running.** Everything else adapts automatically.

| Parameter | What it controls |
|-----------|------------------|
| `HAS_TOPIC_LABELS` | Set `True` if you have manually labelled `topic_info_labelled.xlsx` |
| `LABEL_COL` | Column name for human topic labels in the labelled file |
| `CATEGORY_COL` | Column name for topic category (optional grouping) |
| `RANDOM_SEED` | Seed for any sampling operations |
| `FIGURES_DIR` | Where to save figures (from config by default) |

In [ ]:
# ── 🔧 Edit these values ──────────────────────────────────────────────────────

# Do you have a manually labelled topic table?
# Expected file: TOPIC_INFO_DIR / "topic_info_labelled.xlsx"
# Required columns: "Topic", LABEL_COL, CATEGORY_COL (optional)
HAS_TOPIC_LABELS  = False
LABEL_COL         = "Topic Label"     # column name in your labelled file
CATEGORY_COL      = "Category"        # set to None if absent

# Random seed for sampling
RANDOM_SEED = 42

---
# 2. Load data

Required inputs come directly from the earlier pipeline notebooks. Optional inputs are loaded only if the relevant flags are set.

In [ ]:
# Documents
print("Loading documents...")
with open(PREPROCESSED_DOCS_PATH, "rb") as f:
    docs = pickle.load(f)
print(f"  ✓ {len(docs):,} documents")

# Embeddings
print("Loading embeddings (memory-mapped)...")
embeddings = np.load(EMBEDDINGS_PATH, mmap_mode="r")
print(f"  ✓ {embeddings.shape}")

# Topic model
print("Loading topic model...")
topic_model = BERTopic.load(str(TOPIC_MODEL_PATH))
print(f"  ✓ {len(topic_model.get_topic_info())} topics")

# Topic assignments
topics = np.load(TOPICS_ASSIGNED_PATH)
print(f"  ✓ Topic assignments loaded ({len(topics):,})")

# Results CSV
results_df = pd.read_csv(TOPIC_MODEL_RESULTS_PATH)
print(f"  ✓ Results CSV: {len(results_df):,} rows")

# Topic info
topic_info = pd.read_csv(TOPIC_INFO_PATH)
print(f"  ✓ Topic info: {len(topic_info)} rows")

print("\n✅ All required data loaded.")

In [23]:
# Optional: manually labelled topic table
topic_labels_df = None  # always defined

if HAS_TOPIC_LABELS:
    if Path(LABELLED_INFO_PATH).exists():
        topic_labels_df = pd.read_excel(LABELLED_INFO_PATH)
        print(f"✓ Labelled topic table loaded – {len(topic_labels_df)} rows")
        print(f"  Columns: {list(topic_labels_df.columns)}")
    else:
        print(f"⚠️  HAS_TOPIC_LABELS=True but file not found.")
        HAS_TOPIC_LABELS = False

# Build a topic size lookup from results_df (accurate full-corpus counts)
topic_size = results_df.groupby("topic").size().to_dict()

def get_label(topic_id: int) -> str:
    # Prefer manual label if available
    if topic_labels_df is not None:
        row = topic_labels_df[topic_labels_df["Topic"] == topic_id]
        if len(row):
            return row[LABEL_COL].values[0]
    # Fall back to BERTopic's auto-generated name from topic_info
    row = topic_info[topic_info["Topic"] == topic_id]
    if len(row) and "Name" in topic_info.columns:
        return row["Name"].values[0]
    return f"Topic {topic_id}"

def get_category(topic_id: int) -> str:
    if topic_labels_df is None or CATEGORY_COL is None:
        return "—"
    row = topic_labels_df[topic_labels_df["Topic"] == topic_id]
    return row[CATEGORY_COL].values[0] if len(row) else "—"

def get_size(topic_id: int) -> int:
    return int(topic_size.get(topic_id, 0))

---
# 3. Basic checks

In [ ]:
n_docs      = len(docs)
n_outliers  = int((topics == -1).sum())
n_topics    = len(np.unique(topics[topics != -1]))

print("=" * 55)
print("CORPUS SUMMARY")
print("=" * 55)
print(f"  Documents (total)     : {n_docs:,}")
print(f"  Topics (excl. -1)     : {n_topics}")
print(f"  Outliers (topic -1)   : {n_outliers:,} "
      f"({n_outliers / n_docs * 100:.2f}%)")
print(f"  Embeddings shape      : {embeddings.shape}")
print(f"  Docs / embeddings OK  : {n_docs == embeddings.shape[0]}")

In [ ]:
# Raw topic_info from the model
topic_info.head(20)

In [ ]:
# Topic size distribution (non-outlier topics only)
unique_ids, counts = np.unique(topics, return_counts=True)
non_out_counts     = counts[unique_ids != -1]

print("Topic size distribution (non-outlier topics):")
print(f"  Min    : {non_out_counts.min():,}")
print(f"  Median : {int(np.median(non_out_counts)):,}")
print(f"  Mean   : {non_out_counts.mean():.0f}")
print(f"  Max    : {non_out_counts.max():,}")
print(f"  Std    : {non_out_counts.std():.0f}")

fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(non_out_counts, bins=40, color="steelblue", edgecolor="white", alpha=0.85)
ax.axvline(np.median(non_out_counts), color="orange", linewidth=2,
           linestyle="--", label=f"Median: {int(np.median(non_out_counts)):,}")
ax.set_xlabel("Topic size (documents)")
ax.set_ylabel("Number of topics")
ax.set_title("Topic size distribution", fontweight="bold")
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(FIGURES_DIR / "topic_size_distribution.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
# Scrollable topic label table (only meaningful if HAS_TOPIC_LABELS=True)
if HAS_TOPIC_LABELS:
    from IPython.display import HTML
    cols = ["Topic", LABEL_COL] + ([CATEGORY_COL] if CATEGORY_COL else [])
    tbl  = topic_labels_df[cols].sort_values("Topic")
    display(HTML(
        f'<div style="height:400px;overflow-y:scroll">{tbl.to_html(index=False)}</div>'
    ))
else:
    print("ℹ️  No labelled topic table loaded. "
          "Set HAS_TOPIC_LABELS=True and provide topic_info_labelled.xlsx.")

---
# 4. Topic explorer

Use `explore_topic(topic_id)` to inspect any individual topic. It prints:
- Size and percentage of corpus
- `n_top_words`: Adjust to print top X words by frequency
- BERTopic top-word representation
- `n_samples`: Adjust to print X sample of documents

If you have labels loaded, the topic label and category are shown at the top.

In [28]:
def _word_freq(doc_list: list, n: int = 25) -> Counter:
    return Counter(" ".join(doc_list).lower().split()).most_common(n)


def explore_topic(
    topic_id: int,
    n_samples: int = 10,
    n_top_words: int = 25,
    random_seed: int = RANDOM_SEED,
) -> None:
    random.seed(random_seed)
    doc_indices = np.where(topics == topic_id)[0]
    topic_docs  = [docs[i] for i in doc_indices]

    # Label: manual → topic_info Name → fallback
    if topic_labels_df is not None:
        row = topic_labels_df[topic_labels_df["Topic"] == topic_id]
        label = row[LABEL_COL].values[0] if len(row) else f"Topic {topic_id} (unlabelled)"
    else:
        row = topic_info[topic_info["Topic"] == topic_id]
        label = row["Name"].values[0] if len(row) else f"Topic {topic_id}"

    # Category: only from manual labels
    category = "—"
    if topic_labels_df is not None and CATEGORY_COL:
        row = topic_labels_df[topic_labels_df["Topic"] == topic_id]
        category = row[CATEGORY_COL].values[0] if len(row) else "—"

    print(f"\n{'=' * 65}")
    print(f"TOPIC {topic_id}: {label}")
    if category != "—":
        print(f"Category : {category}")
    print(f"Size     : {len(topic_docs):,} docs  "
          f"({len(topic_docs) / len(docs) * 100:.2f}% of corpus)")
    print(f"{'=' * 65}")

    # BERTopic top-word representation
    btopic_words = topic_model.get_topic(topic_id)
    if btopic_words:
        print("\nBERTopic representation (word, score):")
        for word, score in btopic_words[:10]:
            print(f"  {word:<25} {score:.4f}")

    # Frequency-based top words
    print(f"\nTop {n_top_words} words by frequency:")
    for word, count in _word_freq(topic_docs, n_top_words):
        print(f"  {word:<30} {count:>8,}")

    # Random document sample
    print(f"\n{n_samples} random documents:")
    for i, doc in enumerate(
        random.sample(topic_docs, min(n_samples, len(topic_docs))), 1
    ):
        print(f"  [{i:>2}] {doc}")

In [ ]:
# Change the topic_id to any topic you want to inspect
explore_topic(0, n_samples=10)

---
# 5. Outlier inspection

Qualitative assessment of documents that remain in topic `-1`.

The inspection distinguishes between:
- **Noise**: very short, idiosyncratic documents with unique vocabulary
- **Uncaptured themes**: outliers that share vocabulary (potential missed topics)

In [30]:
# Returns a dict with 'outlier_indices', 'outlier_docs', 'word_counts', and 'lengths' for downstream use
def inspect_outliers(
    topics: np.ndarray,
    docs: list,
    n_samples: int = 20,
    n_top_words: int = 30,
    random_seed: int = RANDOM_SEED,
) -> dict:
    random.seed(random_seed)
    outlier_idx  = np.where(topics == -1)[0]
    out_docs     = [docs[i] for i in outlier_idx]
    non_out_docs = [docs[i] for i in np.where(topics != -1)[0]]
    total, n_out = len(topics), len(outlier_idx)

    print("=" * 65)
    print("OUTLIER INSPECTION")
    print("=" * 65)
    print(f"Total documents  : {total:,}")
    print(f"Final outliers   : {n_out:,}  ({n_out / total * 100:.2f}%)")

    # Document length
    lengths     = [len(d.split()) for d in out_docs]
    non_lengths = [len(d.split()) for d in non_out_docs]
    print("\n── Document length (words) ──")
    print(f"  Outlier   – mean: {np.mean(lengths):.1f}, "
          f"median: {np.median(lengths):.1f}, "
          f"min/max: {min(lengths)}/{max(lengths)}")
    print(f"  Non-outlier – mean: {np.mean(non_lengths):.1f}")

    # Top words
    word_counts = Counter(" ".join(out_docs).lower().split())
    print(f"\n── Top {n_top_words} words in outlier pool ──")
    for word, count in word_counts.most_common(n_top_words):
        print(f"  {word:<30} {count:>8,}  ({count / n_out * 100:.1f}%)")

    # Hapax / idiosyncrasy check
    hapax = [d for d in out_docs
             if all(word_counts[w] == 1 for w in d.lower().split())]
    print("\n── Idiosyncrasy check ──")
    print(f"  Docs where ALL words appear only once: "
          f"{len(hapax):,}  ({len(hapax) / n_out * 100:.1f}%)")
    print("  → High (> 50%) = mostly noise")
    print("  → Low  (< 20%) = shared vocabulary → possible uncaptured theme")

    # Very short docs
    short = [d for d in out_docs if len(d.split()) <= 2]
    print("\n── Very short outlier docs (≤ 2 words) ──")
    print(f"  Count : {len(short):,}  ({len(short) / n_out * 100:.1f}%)")
    for d in random.sample(short, min(10, len(short))):
        print(f"    {d}")

    # Random document sample
    print(f"\n── {n_samples} random outlier documents ──")
    for i, doc in enumerate(
        random.sample(out_docs, min(n_samples, n_out)), 1
    ):
        print(f"  [{i:>2}] ({len(doc.split())} words) {doc}")

    return {
        "outlier_indices": outlier_idx,
        "outlier_docs":    out_docs,
        "word_counts":     word_counts,
        "lengths":         lengths,
    }

In [ ]:
outlier_results = inspect_outliers(
    topics=topics,
    docs=docs,
    n_samples=20,
    n_top_words=30,
)

---
# 6. Outlier plausibility

For a sample of outlier documents, find their **nearest existing topic** via cosine similarity between the document embedding and each topic's centroid embedding. This distinguishes:

- **Plausible outliers** (similarity ≥ threshold): the document is close to an existing topic and may have been correctly rejected or could be re-assigned
- **Genuine noise** (similarity < threshold): the document is far from all topics

Use the borderline analysis to spot topics that could absorb many outliers if the threshold were relaxed.

In [32]:
# Cosine-similarity assessment of outlier plausibility
def assess_outlier_plausibility(
    topic_model: BERTopic,
    docs: list,
    embeddings: np.ndarray,
    topics: np.ndarray,
    n_sample: int = 200,
    similarity_threshold: float = 0.3, # Minimum cosine similarity to determine plausible outliers
    label_lookup: callable = None,
    random_seed: int = RANDOM_SEED,
) -> tuple:

    random.seed(random_seed)
    np.random.seed(random_seed)

    out_idx  = np.where(topics == -1)[0]
    out_embs = embeddings[out_idx]
    out_docs = [docs[i] for i in out_idx]
    print(f"Total outliers : {len(out_idx):,}  |  shape: {out_embs.shape}")

    # Sample (if needed)
    if len(out_docs) > n_sample:
        idx      = np.random.choice(len(out_docs), n_sample, replace=False)
        s_docs   = [out_docs[i] for i in idx]
        s_embs   = out_embs[idx]
    else:
        s_docs, s_embs = out_docs, out_embs

    # Topic centroid embeddings (row 0 is topic -1 → skip)
    topic_ids   = sorted(t for t in topic_model.get_topics() if t != -1)
    topic_embs  = np.array(topic_model.topic_embeddings_)[1:]

    sims          = cosine_similarity(s_embs, topic_embs)
    best_idx      = sims.argmax(axis=1)
    best_sim      = sims.max(axis=1)
    best_topic_id = [topic_ids[i] for i in best_idx]

    # Labels, if present
    if label_lookup is None:
        label_lookup = lambda tid: f"Topic {tid}"
    best_label = [label_lookup(t) for t in best_topic_id]

    results = (
        pd.DataFrame({
            "document":      s_docs,
            "nearest_topic": best_topic_id,
            "nearest_label": best_label,
            "similarity":    best_sim.round(3),
            "plausible":     best_sim >= similarity_threshold,
        })
        .sort_values("similarity", ascending=False)
        .reset_index(drop=True)
    )

    n_plaus = results["plausible"].sum()
    print(f"\n{'=' * 65}")
    print(f"OUTLIER PLAUSIBILITY  (n={len(results)}, threshold={similarity_threshold})")
    print(f"{'=' * 65}")
    print(f"  Plausibly near existing topic : {n_plaus} ({n_plaus/len(results)*100:.1f}%)")
    print(f"  Genuinely unassignable        : "
          f"{len(results)-n_plaus} ({(len(results)-n_plaus)/len(results)*100:.1f}%)")

    print("\n── Similarity thresholds ──")
    for thr in [0.1, 0.2, 0.3, 0.4, 0.5]:
        n = int((best_sim >= thr).sum())
        print(f"  >= {thr} : {n:>4}  ({n/len(results)*100:.1f}%)")

    print("\n── Most common nearest topics ──")
    for lbl, cnt in results["nearest_label"].value_counts().head(10).items():
        print(f"  {lbl:<45} {cnt:>4}  ({cnt/len(results)*100:.1f}%)")

    print("\n── High-similarity examples (plausible) ──")
    for _, row in results[results["plausible"]].head(8).iterrows():
        print(f"  [{row['similarity']:.3f}] → {row['nearest_label']}")
        print(f"    {row['document']}")

    print("\n── Low-similarity examples (noise) ──")
    for _, row in results[results["similarity"] < 0.1].head(8).iterrows():
        print(f"  [{row['similarity']:.3f}] → {row['nearest_label']}")
        print(f"    {row['document']}")

    return results, sims, topic_ids

In [ ]:
plaus_results, plaus_sims, plaus_topic_ids = assess_outlier_plausibility(
    topic_model=topic_model,
    docs=docs,
    embeddings=embeddings,
    topics=topics,
    n_sample=200,
    similarity_threshold=0.3,
    label_lookup=get_label,
)

In [ ]:
# Print borderline outlier cases with their top-3 nearest topic candidates.
def show_borderline_outliers(
    results: pd.DataFrame,
    sims: np.ndarray,
    topic_ids: list,
    label_lookup: callable = None,
    threshold_low: float = 0.3, 
    threshold_high: float = 0.5,
) -> None:
    if label_lookup is None:
        label_lookup = lambda tid: f"Topic {tid}"

    mask = (results["similarity"] >= threshold_low) & \
           (results["similarity"] <  threshold_high)
    borderline = results[mask]

    print(f"Borderline cases ({threshold_low} ≤ sim < {threshold_high}): "
          f"{len(borderline)}")
    print("=" * 65)
    for idx, row in borderline.iterrows():
        top3_i  = sims[idx].argsort()[::-1][:3]
        top3    = [(label_lookup(topic_ids[i]), round(float(sims[idx][i]), 3))
                   for i in top3_i]
        print(f"  [{row['similarity']:.3f}] → {row['nearest_label']}")
        print(f"    Doc   : {row['document']}")
        print(f"    Top 3 : {top3}")
        print()


show_borderline_outliers(
    results=plaus_results,
    sims=plaus_sims,
    topic_ids=plaus_topic_ids,
    label_lookup=get_label,
    threshold_low=0.3,
    threshold_high=0.5,
)

---
# 7. Topic quality & concentration

Two complementary views of whether the topic solution is well-balanced:

- [**Gini coefficient**](https://medium.com/@kstarun/model-evaluation-metrics-gini-coefficient-db919ed09306): 0 = all topics equal size; 1 = one topic dominates everything
- **Size anomalies**: topics below or above configurable size thresholds

In [ ]:
def gini_coefficient(counts: np.ndarray) -> float:
    s = np.sort(counts.astype(float))
    n = len(s)
    return (2 * np.sum(np.arange(1, n + 1) * s)) / (n * s.sum()) - (n + 1) / n


unique_ids, counts = np.unique(topics, return_counts=True)
non_out_mask       = unique_ids != -1
non_out_ids        = unique_ids[non_out_mask]
non_out_counts     = counts[non_out_mask]

gini = gini_coefficient(non_out_counts)
print("=" * 55)
print("TOPIC CONCENTRATION")
print("=" * 55)
print(f"  Gini coefficient : {gini:.3f}")
print("  (0 = all topics equal size; 1 = extreme inequality)")
print(f"  Imbalance ratio  : {non_out_counts.max() / non_out_counts.min():.1f}×  "
      f"(largest / smallest)")

In [ ]:
# Configurable thresholds ────────────────────────────────────────────────────
SMALL_THRESHOLD = 100   # topics with fewer docs than this are flagged
LARGE_PERCENTILE = 95   # topics above this percentile are flagged as dominant

large_threshold = np.percentile(non_out_counts, LARGE_PERCENTILE)

topic_dist = (
    pd.DataFrame({"Topic": non_out_ids, "Count": non_out_counts})
    .assign(Label=lambda d: d["Topic"].map(get_label))
    .sort_values("Count", ascending=False)
)

small_topics = topic_dist[topic_dist["Count"] < SMALL_THRESHOLD]
large_topics = topic_dist[topic_dist["Count"] > large_threshold]

print(f"Topics with < {SMALL_THRESHOLD} docs : {len(small_topics)}")
if len(small_topics):
    print(small_topics.to_string(index=False))

print(f"\nTopics above {LARGE_PERCENTILE}th percentile "
      f"(> {large_threshold:.0f} docs) : {len(large_topics)}")
print(large_topics.head(10).to_string(index=False))

---
# 8. Fragmentation & overlap

### `find_overlapping_topics`

[Jaccard-based](https://www.ibm.com/think/topics/jaccard-similarity#:~:text=Jaccard%20similarity%20is%20a%20statistical,that%20the%20sets%20are%20identical.) overlap check within a BERTopic model.

| Parameter | Description |
|-----------|-------------|
| `model` | Fitted BERTopic model |
| `top_n` | Number of top words per topic to compare |
| `similarity_threshold` | Minimum Jaccard score to flag a pair |
| `label_lookup` | Optional callable mapping `topic_id → label string` |

**Returns** a DataFrame of overlapping topic pairs sorted by Jaccard score descending, with columns `topic_a`, `topic_b`, `label_topic_a`, `label_topic_b`, `jaccard`, `shared_words`.

In [ ]:
def find_overlapping_topics(
    model: BERTopic,
    top_n: int = 20,
    similarity_threshold: float = 0.3,
    label_lookup: callable = None,
) -> pd.DataFrame:

    if label_lookup is None:
        label_lookup = lambda tid: f"Topic {tid}"

    def _top_words(tid):
        words = model.get_topic(tid)
        return set(w for w, _ in words[:top_n]) if words else set()

    ids  = [t for t in model.get_topic_info()["Topic"] if t != -1]
    rows = []

    for i, ia in enumerate(ids):
        wa = _top_words(ia)
        if not wa:
            continue
        for ib in ids[i + 1:]:
            wb = _top_words(ib)
            if not wb:
                continue
            inter = wa & wb
            # Jaccard similarity calculation
            j = len(inter) / len(wa | wb)
            if j >= similarity_threshold:
                rows.append({
                    "topic_a":       ia,
                    "topic_b":       ib,
                    "label_topic_a": label_lookup(ia),
                    "label_topic_b": label_lookup(ib),
                    "jaccard":       round(j, 3),
                    "shared_words":  ", ".join(sorted(inter)),
                })

    result = (
        pd.DataFrame(rows).sort_values("jaccard", ascending=False)
        if rows else pd.DataFrame()
    )
    print(f"Overlapping topic pairs: {len(result)}")
    return result


overlap_df = find_overlapping_topics(
    model=topic_model,
    top_n=20,
    similarity_threshold=0.3,
    label_lookup=get_label,
)

In [ ]:
# Show top overlapping topic pairs
# Jaccard similarity ranges from 0 - 1, where 0 indicates that the sets have no elements in common, and one indicates that the sets are identical.
print("Top overlapping topic pairs:")
overlap_df.head(10)

---
# 9. BERTopic visualisations

BERTopic ships with several built-in interactive Plotly visualisations. All of them render inline in Jupyter.

| Visualisation | What it shows |
|--------------|---------------|
| `visualize_topics()` | 2D UMAP scatter of topic centroids, sized by topic size |
| `visualize_barchart()` | Top-N topics as horizontal bar charts of top words |
| `visualize_heatmap()` | Topic similarity matrix (cosine similarity between centroids) |
| `visualize_hierarchy()` | Hierarchical clustering dendrogram of topics |
| `visualize_topics_over_time()` | Topic prevalence over time (requires timestamps) |

In [ ]:
# 2D scatter of topic centroids
topic_model.visualize_topics()

In [ ]:
# Top-word bar charts for the N largest topics
topic_model.visualize_barchart(top_n_topics=12)

In [ ]:
# Topic similarity heatmap
topic_model.visualize_heatmap()

In [ ]:
# Hierarchical topic clustering
topic_model.visualize_hierarchy()

## Category treemap *(requires labelled topics with CATEGORY_COL)*

A treemap shows how documents are distributed across manually defined **categories** (higher-level groupings of topics). Each box is sized by the total number of documents in that category.

Requires `HAS_TOPIC_LABELS=True` and `CATEGORY_COL` to be set.

In [ ]:
if HAS_TOPIC_LABELS and CATEGORY_COL and SQUARIFY_AVAILABLE:
    # Build category→count mapping from the topic assignments
    topic_counts = (
        pd.Series(topics[topics != -1])
        .value_counts()
        .rename_axis("Topic")
        .reset_index(name="Count")
    )
    cat_df = (
        topic_counts
        .merge(topic_labels_df[["Topic", CATEGORY_COL]], on="Topic", how="left")
        .groupby(CATEGORY_COL)["Count"].sum()
        .reset_index()
        .sort_values("Count", ascending=False)
    )
    total_cat = cat_df["Count"].sum()
    cat_df["Pct"] = cat_df["Count"] / total_cat * 100

    colors = sns.color_palette("tab20", n_colors=len(cat_df))
    labels = [
        f"{row[CATEGORY_COL]}\n{row['Pct']:.1f}%"
        for _, row in cat_df.iterrows()
    ]

    fig, ax = plt.subplots(figsize=(18, 12))
    squarify.plot(
        sizes=cat_df["Pct"],
        label=labels,
        color=colors,
        alpha=0.85,
        ax=ax,
        pad=True,
        text_kwargs={"fontsize": 11, "fontweight": "bold", "color": "white"},
    )
    ax.axis("off")
    ax.set_title("Topic category distribution", fontsize=18, fontweight="bold", pad=20)
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / "category_treemap.png", dpi=300, bbox_inches="tight")
    plt.show()

elif not HAS_TOPIC_LABELS:
    print("ℹ️  Treemap requires HAS_TOPIC_LABELS=True and CATEGORY_COL to be set.")
elif not SQUARIFY_AVAILABLE:
    print("ℹ️  Install squarify to enable the treemap: pip install squarify")

---
# 10. Topic coherence (C_v)

C_v coherence measures how semantically related the top words within each topic are, using a sliding-window co-occurrence + NPMI + cosine similarity pipeline ([Röder et al., 2015](https://svn.aksw.org/papers/2015/WSDM_Topic_Evaluation/public.pdf)).

A higher score means the top words genuinely co-occur and are thematically tight. Typical ranges on social-media corpora are 0.3–0.7.

This section:
1. Computes per-topic C_v across all non-outlier topics
2. Visualises the distribution
3. Flags low-coherence and size-anomalous topics
4. If human labels are loaded: compares *Incoherent*-labelled topics against the rest


In [ ]:
if not GENSIM_AVAILABLE:
    print("⏭️  gensim not installed – skipping Section X.")
else:
    print("Tokenising documents...")
    tokenized_docs = [doc.lower().split() for doc in docs]
    dictionary     = Dictionary(tokenized_docs)
    print(f"  ✓ Vocabulary size: {len(dictionary):,} tokens")

    # Build per-topic top-10 word lists from the full topic assignment
    # (not just the training sample — ensures counts reflect full corpus)
    topic_words_coh: list = []
    topic_ids_coh:   list = []

    for tid in sorted(t for t in np.unique(topics) if t != -1):
        doc_idx    = np.where(topics == tid)[0]
        topic_docs = [docs[i] for i in doc_idx]
        wc         = Counter(" ".join(topic_docs).lower().split())
        top_words  = [w for w, _ in wc.most_common(10)]
        topic_words_coh.append(top_words)
        topic_ids_coh.append(tid)

    print(f"  ✓ Topics prepared: {len(topic_words_coh)}")

In [ ]:
if GENSIM_AVAILABLE:
    # ⚠️ Runtime note: Building the co-occurrence dictionary and running CoherenceModel can take several minutes on large corpora.
    print("Computing C_v coherence...")
    coherence_model = CoherenceModel(
        topics=topic_words_coh,
        texts=tokenized_docs,
        dictionary=dictionary,
        coherence="c_v",
        processes=4,
    )
    coherence_score     = coherence_model.get_coherence()
    coherence_per_topic = coherence_model.get_coherence_per_topic()

    print(f"\nC_v (overall mean) : {coherence_score:.4f}")
    print(f"Mean per topic     : {np.mean(coherence_per_topic):.4f}")
    print(f"Median             : {np.median(coherence_per_topic):.4f}")
    print(f"Std dev            : {np.std(coherence_per_topic):.4f}")
    print(f"Min / Max          : {min(coherence_per_topic):.4f} / "
          f"{max(coherence_per_topic):.4f}")

In [ ]:
if GENSIM_AVAILABLE:
    mean_c   = np.mean(coherence_per_topic)
    median_c = np.median(coherence_per_topic)

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    axes[0].hist(coherence_per_topic, bins=30, color="steelblue",
                 edgecolor="white", alpha=0.85)
    axes[0].axvline(mean_c,   color="red",    linestyle="--", linewidth=2,
                    label=f"Mean: {mean_c:.3f}")
    axes[0].axvline(median_c, color="orange", linestyle="--", linewidth=2,
                    label=f"Median: {median_c:.3f}")
    axes[0].set_xlabel("Coherence (C_v)")
    axes[0].set_ylabel("Number of topics")
    axes[0].set_title("Distribution of topic coherence scores", fontweight="bold")
    axes[0].legend()
    axes[0].grid(alpha=0.3)

    bp = axes[1].boxplot(coherence_per_topic, patch_artist=True)
    bp["boxes"][0].set_facecolor("lightblue")
    axes[1].set_ylabel("Coherence (C_v)")
    axes[1].set_title("Topic coherence – box plot", fontweight="bold")
    axes[1].grid(alpha=0.3)

    plt.tight_layout()
    plt.savefig(FIGURES_DIR / "topic_coherence_distribution.png",
                dpi=300, bbox_inches="tight")
    plt.show()

In [ ]:
if GENSIM_AVAILABLE:
    # Build coherence DataFrame with labels if available
    coherence_df = pd.DataFrame({
        "Topic":     topic_ids_coh,
        "Coherence": coherence_per_topic,
        "TopWords":  [" ".join(w[:5]) for w in topic_words_coh],
        "Size":      [
            int((topics == tid).sum()) for tid in topic_ids_coh
        ],
    })

    if HAS_TOPIC_LABELS:
        coherence_df = coherence_df.merge(
            topic_labels_df[["Topic", LABEL_COL]
                            + ([CATEGORY_COL] if CATEGORY_COL else [])],
            on="Topic",
            how="left",
        )

    coherence_df = coherence_df.sort_values("Coherence", ascending=False)
    print(f"Coherence DataFrame: {len(coherence_df)} rows")
    coherence_df.head(10)

In [ ]:
# Print rows of most coherent & incoherent topics
if GENSIM_AVAILABLE:
    def _print_coh_rows(df: pd.DataFrame, title: str, n: int = 10) -> None:
        label_col = LABEL_COL if (HAS_TOPIC_LABELS and LABEL_COL in df.columns) else None
        print(f"\n{title}")
        print("=" * 90)
        for _, row in df.head(n).iterrows():
            lbl = row[label_col] if label_col else f"Topic {int(row['Topic'])}"
            print(f"  {lbl:<40}  Coh: {row['Coherence']:.3f}  "
                  f"Size: {int(row['Size']):>8,}  "
                  f"Top words: {row['TopWords']}")

    _print_coh_rows(coherence_df,                        "🏆 MOST COHERENT")
    _print_coh_rows(coherence_df.sort_values("Coherence"), "🚫 LEAST COHERENT")

In [ ]:
# Print low coherence / incoherent topics
if GENSIM_AVAILABLE:
    low_thr   = float(np.percentile(coherence_per_topic, 25))
    med_thr   = float(np.median(coherence_per_topic))
    small_thr = int(np.percentile(coherence_df["Size"], 25))
    large_thr = int(np.percentile(coherence_df["Size"], 75))

    print(f"Thresholds – low coherence : {low_thr:.3f}  |  median: {med_thr:.3f}")
    print(f"             small size    : {small_thr:,}  |  large: {large_thr:,}")

    def _flag_group(title, mask):
        subset = coherence_df[mask]
        print(f"\n{title}  →  {len(subset)} topics")
        print("=" * 90)
        for _, row in subset.iterrows():
            lbl = (row[LABEL_COL] if (HAS_TOPIC_LABELS and LABEL_COL in row)
                   else f"Topic {int(row['Topic'])}")
            print(f"  {lbl:<40}  Coh: {row['Coherence']:.3f}  "
                  f"Size: {int(row['Size']):>8,}")

    _flag_group(
        f"1 · Low coherence (< {low_thr:.3f})",
        coherence_df["Coherence"] < low_thr,
    )
    _flag_group(
        f"2 · Large & incoherent (size > {large_thr:,}, coh < {med_thr:.3f})",
        (coherence_df["Size"] > large_thr) & (coherence_df["Coherence"] < med_thr),
    )
    _flag_group(
        f"3 · Small & incoherent (size < {small_thr:,}, coh < {med_thr:.3f})",
        (coherence_df["Size"] < small_thr) & (coherence_df["Coherence"] < med_thr),
    )

In [ ]:
# Compare topics labelled 'Incoherent' vs labelled topics
# Only runs if HAS_TOPIC_LABELS=True and LABEL_COL contains 'Incoherent' entries
if GENSIM_AVAILABLE and HAS_TOPIC_LABELS and LABEL_COL in coherence_df.columns:
    is_inc   = coherence_df[LABEL_COL].str.contains("Incoherent", case=False, na=False)
    inc_df   = coherence_df[is_inc]
    coh_df_l = coherence_df[~is_inc]

    if len(inc_df) > 0 and len(coh_df_l) > 0:
        coh_scores  = coh_df_l["Coherence"]
        incoh_scores = inc_df["Coherence"]

        print(f"Topics labelled Incoherent : {len(inc_df)}  "
              f"(mean C_v = {incoh_scores.mean():.3f})")
        print(f"Topics with meaningful label: {len(coh_df_l)}  "
              f"(mean C_v = {coh_scores.mean():.3f})")

        _, p_lev = levene(coh_scores, incoh_scores)
        t, p_t   = ttest_ind(coh_scores, incoh_scores, equal_var=(p_lev > 0.05))
        pooled   = np.sqrt((coh_scores.std()**2 + incoh_scores.std()**2) / 2)
        d        = (coh_scores.mean() - incoh_scores.mean()) / pooled
        effect   = ("negligible" if abs(d) < 0.2 else
                    "small" if abs(d) < 0.5 else
                    "medium" if abs(d) < 0.8 else "large")

        test_lbl = "Welch's t-test" if p_lev <= 0.05 else "t-test"
        print(f"\n{test_lbl}: t = {t:.4f},  p = {p_t:.6f}  "
              f"{'✅ significant' if p_t < 0.05 else '✗ not significant'}")
        print(f"Cohen's d = {d:.3f}  ({effect} effect)")

        fig, ax = plt.subplots(figsize=(8, 5))
        ax.hist(coh_scores,   bins=25, alpha=0.6, color="steelblue",
                edgecolor="white",
                label=f"Labelled (mean={coh_scores.mean():.3f}, n={len(coh_scores)})")
        ax.hist(incoh_scores, bins=25, alpha=0.6, color="coral",
                edgecolor="white",
                label=f"Incoherent (mean={incoh_scores.mean():.3f}, n={len(incoh_scores)})")
        ax.axvline(coh_scores.mean(),   color="blue", linestyle="--", linewidth=2)
        ax.axvline(incoh_scores.mean(), color="red",  linestyle="--", linewidth=2)
        ax.set_xlabel("Coherence (C_v)")
        ax.set_ylabel("Number of topics")
        ax.set_title("C_v: labelled vs. incoherent topics", fontweight="bold")
        ax.legend()
        ax.grid(alpha=0.3)
        plt.tight_layout()
        plt.savefig(FIGURES_DIR / "coherence_labelled_vs_incoherent.png",
                    dpi=300, bbox_inches="tight")
        plt.show()
    else:
        print("ℹ️  No 'Incoherent' labels found in label column – skipping comparison.")
else:
    print("ℹ️  Label-based coherence comparison requires HAS_TOPIC_LABELS=True.")

In [ ]:
if GENSIM_AVAILABLE:
    # Save coherence table
    # coherence_df.to_excel(TOPIC_COHERENCE_PATH, index=False)
    # print(f"✅ Saved to {TOPIC_COHERENCE_PATH}")
    print("Uncomment the lines above to save the coherence table.")

---
# 11. Topic diversity

**Unique keyword ratio**: what fraction of all top-10 keyword slots across topics are unique?

- **1.0** = no two topics share any top keyword
- **Low score** = many topics share keyword

In [ ]:
# Build top-10 keyword lists from BERTopic representations
# (uses the model's own c-TF-IDF words, not frequency counts)
all_topic_words = []
for tid in sorted(t for t in np.unique(topics) if t != -1):
    words = topic_model.get_topic(tid)
    if words:
        all_topic_words.extend([w for w, _ in words[:10]])

unique_kw = set(all_topic_words)
diversity  = len(unique_kw) / len(all_topic_words) if all_topic_words else 0.0

print("=" * 45)
print("TOPIC DIVERSITY")
print("=" * 45)
print(f"  Total keyword slots : {len(all_topic_words):,}")
print(f"  Unique keywords     : {len(unique_kw):,}")
print(f"  Diversity score     : {diversity:.4f}  "
      f"({diversity * 100:.1f}% unique)")
print()
print("  Interpretation:")
if diversity >= 0.8:
    print("  ✅ High diversity – topics have distinct vocabularies")
elif diversity >= 0.5:
    print("  ⚠️  Moderate diversity – some vocabulary overlap across topics")
else:
    print("  ❌ Low diversity – many shared keywords; "
          "check for fragmentation or overly generic topics")

# Also report coherence-based diversity if gensim was run
if GENSIM_AVAILABLE:
    all_coh_words = [w for wlist in topic_words_coh for w in wlist]
    unique_coh    = set(all_coh_words)
    div_freq      = len(unique_coh) / len(all_coh_words) if all_coh_words else 0.0
    print(f"\n  Frequency-based diversity: {div_freq:.4f}  "
          f"({div_freq * 100:.1f}% unique)")

---
# AI disclosure statement

AI tools were used to assist:
- developing, labelling, and debugging code
- formatting Markdown cells

AI tools used:
- [CursorAI (Desktop version)](https://cursor.com/agents)
- [Claude AI](https://claude.ai/)
- [ChatGPT](https://chatgpt.com/)

I acknowledge my responsibility as a researcher to thoroughly verify all outputs and content produced by AI tools and accept full accountability for their accuracy and validity.

XXX

---
# References
Röder, M., Both, A., & Hinneburg, A. (2015). Exploring the space of topic coherence measures. Proceedings of the 8th ACM International Conference on Web Search and Data Mining (WSDM '15), 399–408. https://doi.org/10.1145/2684822.2685324